In [ ]:
import streamlit as st

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import joblib



# setup data

In [36]:
df = pd.read_csv("data/data.csv")
df = df[df["Disaster Type"] == "Flood"]
df = df[df["Start Year"] > 1966]
df  = df[["Start Year", "CPI", "Total Damage, Adjusted ('000 US$)"]]
df = df.dropna(subset = "Total Damage, Adjusted ('000 US$)")
df[["Sum"]] = df.groupby(["Start Year"])[["CPI"]].transform("count")
df["Total Damage, Adjusted ('000 US$)"] = df.groupby(["Start Year"])["Total Damage, Adjusted ('000 US$)"].transform("sum")
df["cumsum"] = df["Sum"].cumsum()
df


,Start Year,CPI,"Total Damage, Adjusted ('000 US$)",Sum,cumsum
43,1990,44.645877,1624.0,1,1
49,1991,46.536621,100351.0,3,4
51,1991,46.536621,100351.0,3,7
53,1991,46.536621,100351.0,3,10
59,1992,47.946130,99487.0,1,11
66,1993,49.361332,20259.0,1,12
72,1994,50.648403,406726.0,1,13
74,1995,52.069299,165165.0,1,14
82,1996,53.595561,282486.0,2,16
84,1996,53.595561,282486.0,2,18


# Model du doan CPI so voi nam

In [42]:

X=df[["Start Year"]]
y=df["CPI"]


#divide train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

#train
model_predCPI_year = LinearRegression()
model_predCPI_year.fit(X_train, y_train)

#evaluate
y_pred = model_predCPI_year.predict(X_test)
mse=mean_squared_error(y_test,y_pred)
r2 = r2_score(y_test,y_pred)
print(f"R² = {r2:.3f}")
print(f"Hệ số: {model_predCPI_year.coef_[0]:.3f}")
print(f"Intercept: {model_predCPI_year.intercept_:.3f}")
print(f"MSE: {mse}")

R² = 0.726
Hệ số: 1.514
Intercept: -2967.911
MSE: 5.870038999597592


# model dự đoán CPI so với số lượng bão

In [ ]:

X=df[["cumsum"]]
y=df["CPI"]

#divide
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#train
model_predCPI_floodAmount = LinearRegression()
model_predCPI_floodAmount.fit(X_train, y_train)

#evaluate
y_pred = model_predCPI_floodAmount.predict(X_test)
mse=mean_squared_error(y_test,y_pred)
r2 = r2_score(y_test,y_pred)


print(f"R² = {r2:.3f}")
print(f"Hệ số: {model_predCPI_floodAmount.coef_[0]:.3f}")
print(f"Intercept: {model_predCPI_floodAmount.intercept_:.3f}")
print(f"MSE: {mse}")

0.9849309795649291
R² = 0.412
Hệ số: 0.286
Intercept: 48.072
MSE: 12.620213083270093


# luu model

In [ ]:
joblib.dump(model_predCPI_floodAmount, "predCPI_floodAmount.pkl")
joblib.dump(model_predCPI_year, "pred_CPI_year.pkl")